# T7 It4: the decision rule around the judge (cut, extra queries, early exit) and the freeze

**Question:** It2 showed the candidate judge removes the fresh-NUDT wrong answers and gains on official under a hand-made cut (smallest zero-wrong
cut + one grid step). Here: which **cut rule** holds up on held-out sets, what happens on the **Jetson** replay (M = 5, extra queries one at a time),
and whether the judge can **stop early** (fewer LoFTR queries = less Jetson time). The rule is then **frozen** on the five seen sets
(`results/frozen_rules_t7.json`, `results/judge_t7.joblib`) before the blind NUDT 606 set is opened (It5).

**Protocol:** leave one set out for everything (judge and cut fitted on the other four sets). Each held-out judge is trained once
(`held_out_probabilities`) and reused by every replay. Gains: paired bootstrap against the agreement-free baselines (PC: bar 8 on the 6 standard
queries, else bar 11 on all 27; Jetson today: bar 8, second pass, cap 10, M = 5).

**Cut rules compared:**
- *zero-wrong + margin k*: the smallest cut with 0 wrong on the training sets at it and at every larger cut, then k grid steps up.
- *Learn-then-Test* (Angelopoulos et al. 2021; conformal risk control): walk the cuts from the strictest down while the one-sided 90 % Clopper-Pearson
  upper bound of the wrong rate among answered training photos stays <= alpha. With 0 wrong in n answers the bound is ~2.3 / n, so the rule also
  refuses cuts that are clean only on little data.

In [ ]:
import os, sys, time, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))   # one OpenMP thread per allowed core (more threads than cores slowed the judge ~100x)
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
from train.checks.candidate_judge import (FEATURES_CTX, FROZEN, FROZEN_VARIANTS, add_context, add_features, baseline, freeze, held_out_probabilities,
                                          leave_one_set_out, paired_gain, summary)
RES = ROOT / "tests/Test_Iteration_7/results"
SETS = ["official", "nudt101", "nudt202", "nudt404", "nudt505"]
table = pd.concat([pd.read_csv(RES / f"signals/{s}.csv.gz") for s in SETS], ignore_index=True)
table = add_context(add_features(table)).reset_index(drop=True)
t0 = time.time(); PROBS = held_out_probabilities(table, SETS, FEATURES_CTX); print("five held-out judges: %.0f s" % (time.time() - t0))
base_pc = baseline(table, 8, 11, min_choice=3)
base_jet = baseline(table, 8, 8, jetson=True, m=5)
display(pd.concat({"PC baseline": summary(base_pc), "Jetson today": summary(base_jet)})[["score", "wrong", "answered", "queries"]])

def run(name, base, **kw):
    s, rows = leave_one_set_out(table, SETS, FEATURES_CTX, probs=PROBS, min_choice=3, override=12, **kw)
    g = paired_gain(base, rows, SETS)
    s["cell"] = s.score.map("{:.2f}".format) + " / " + s.wrong.astype(str) + "w / t" + s.tau.astype(str) + " / " + s.held.map(g)
    return s.assign(rule=name), rows

## 1. Cut rules on the PC (held-out score / wrong / cut / paired gain over the PC baseline)

In [ ]:
out = []
for k in (0, 1, 2):
    out.append(run(f"zero-wrong + margin {k}", base_pc, margin_steps=k)[0])
for a in (0.002, 0.005, 0.01):
    out.append(run(f"Learn-then-Test alpha {a}", base_pc, cut_rule="ltt", alpha=a)[0])
cuts = pd.concat(out)
display(cuts.pivot(index="rule", columns="held", values="cell")[SETS])
cuts.to_csv(RES / "it4_cut_rules_pc.csv", index=False)

**Reading.** alpha 0.002 is too strict (cut 0.99); alpha 0.01 lets the cut fall to 0.6-0.8 and wrong answers come back (101, 202, 505).
alpha 0.005 is the best zero-wrong rule on the PC, but it sits close to that edge.

## 2. The same judge on the Jetson replay: a floor on the cut and a stricter cut for the extra queries

In [ ]:
out, ROWS = [], {}
for name, kw in {"LTT 0.005": dict(), "LTT 0.005, floor 0.9": dict(tau_floor=0.9), "LTT 0.005, floor 0.925": dict(tau_floor=0.925),
                 "LTT 0.005, floor 0.9, extra-query cut 0.95": dict(tau_floor=0.9, wide_tau=0.95),
                 "LTT 0.005, floor 0.9, extra-query cut 0.97": dict(tau_floor=0.9, wide_tau=0.97)}.items():
    s, rows = run(name, base_jet, cut_rule="ltt", alpha=0.005, jetson=True, m=5, **kw)
    out.append(s); ROWS[name] = rows
jet = pd.concat(out)
display(jet.pivot(index="rule", columns="held", values="cell")[SETS])
r = ROWS["LTT 0.005, floor 0.9"]; print("wrong answers left with the floor alone:"); display(r[r.pts < 0])
jet.to_csv(RES / "it4_jetson_rules.csv", index=False)

**Reading.** On the Jetson candidates (M = 5) Learn-then-Test picks lower cuts (0.8-0.85) and wrong answers appear. A floor of 0.9 (the lowest
cut any PC fold chose) removes two; the last one (NUDT 101 photo 101: a 4-inlier look-alike with P 0.93, found on the 9th query) is an
*extra-query* answer. More queries give a look-alike more chances, the same reason today's bars rise from 8 to 11 for the wider queries. A cut of
0.95 for the extra queries removes it, and at that setting no held-out set has a wrong answer.

## 3. Final rule on both replays, with and without the early exit

In [ ]:
final, FINAL_ROWS = [], {}
for variant, kw in FROZEN_VARIANTS.items():
    base = base_jet if kw.get("jetson") else base_pc
    s, rows = run(variant, base, cut_rule="ltt", alpha=FROZEN["alpha"], tau_floor=FROZEN["tau_floor"], wide_tau=FROZEN["wide_tau"], **kw)
    final.append(s); FINAL_ROWS[variant] = rows
final = pd.concat(final)
display(final.pivot(index="rule", columns="held", values="cell").loc[list(FROZEN_VARIANTS)][SETS])
q = final.pivot(index="rule", columns="held", values="queries").loc[list(FROZEN_VARIANTS)][SETS]
print("mean LoFTR queries per photo:"); display(q)
final.to_csv(RES / "it4_final_rule_loso.csv", index=False)
pd.concat([r.assign(variant=v) for v, r in FINAL_ROWS.items()]).to_csv(RES / "it4_final_rule_loso_photos.csv", index=False)

## 4. Time on the Jetson

Model: 2.2 s per LoFTR query at M = 5 (measured in Test_Iteration_6) + the judge's signals (0.93 ms per candidate on one x86 core, i.e. ~5 x 0.93 ms
per query; taken x3 for the Jetson's slower CPU) + the judge itself (200 depth-3 trees, < 1 ms per query). The time classes (stated by the team):
**< 4 s good, 4-6 s viable, >= 6 s slow**, read here as the time a decision rule adds to (or removes from) a photo.

In [ ]:
QUERY_S, SIGNAL_S = 2.2, 3 * 5 * 0.00093 + 0.001
jq = pd.concat({"Jetson today": summary(base_jet)["queries"],
                "jetson (judge)": final[final.rule == "jetson"].set_index("held")["queries"],
                "jetson_early (judge)": final[final.rule == "jetson_early"].set_index("held")["queries"]}, axis=1).loc[SETS]
sec = jq * QUERY_S
for col in ["jetson (judge)", "jetson_early (judge)"]:
    sec[col] += jq[col] * SIGNAL_S
sec.loc["mean"] = sec.mean()
display(sec.round(2).add_suffix(" [s / photo, matching + decision]"))
print("judge overhead per photo: %.0f ms (good)" % (1000 * jq['jetson (judge)'].mean() * SIGNAL_S))
print("early exit saves %.1f s per photo on average" % (sec.loc['mean', 'Jetson today'] - sec.loc['mean', 'jetson_early (judge)']))

## 5. Freeze (fit on all five seen sets) before opening the blind set

In [ ]:
import joblib
frozen = {"rule": {k: v for k, v in FROZEN.items()}, "variants": {}, "seen_sets": SETS,
          "note": "fixed before NUDT fresh 606 (seed 606, pool 2400, photos of fresh 404/505 excluded) was scored"}
for variant in FROZEN_VARIANTS:
    model, tau = freeze(table, SETS, variant)
    frozen["variants"][variant] = dict(tau=tau, **FROZEN_VARIANTS[variant])
    print(variant, "cut", tau)
joblib.dump(model, RES / "judge_t7.joblib")          # the model is the same for every variant (fitted on all labelled candidates)
json.dump(frozen, open(RES / "frozen_rules_t7.json", "w"), indent=1)
print(json.dumps(frozen, indent=1))

## Reading

- The judge with the frozen rule (Learn-then-Test alpha 0.005, floor 0.9, extra-query cut 0.95, override 12) has **0 wrong answers on every
  held-out set**, on the PC and on the Jetson replay.
- **Early exit** (answer after any standard query whose choice has P >= 0.98) cuts LoFTR queries from ~6.3 to ~2 per photo on the Jetson:
  about 9 s saved per photo, for a small score loss. It is the one rule here that changes the time class of the whole photo.
- What is left to show: the same numbers on photos no part of this rule has seen (It5, NUDT 606).